In [ ]:
from pathlib import Path
import sys
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# Add parent directory to path for imports
sys.path.insert(0, str(Path('..').resolve()))

from agents import ZeroShotAgent
from lib.render import render_image
from lib.types import Spec

In [ ]:
# Load canva specs directory
specs_dir = Path('../datasets/canva_specs/')

# Get all spec directories
spec_paths = sorted(list(specs_dir.glob('*/spec.json')))

# Create dataframe
specs_df = pd.DataFrame({
    'template_id': [p.parent.name for p in spec_paths],
    'spec_path': spec_paths
})

print(f"Found {len(specs_df)} design specs")
specs_df.head()

In [ ]:
# Test configuration
TEST_SIZE = 5
EDIT_INSTRUCTION = "make it more dream-like"
OUTPUT_ROOT = Path('../datasets/edits')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Select first N specs for testing
test_specs = specs_df.head(TEST_SIZE).copy()

print(f"Testing on {len(test_specs)} designs")
print(f"Instruction: '{EDIT_INSTRUCTION}'")
print(f"Output directory: {OUTPUT_ROOT}")
print()
print("Test designs:")
for tid in test_specs['template_id']:
    print(f"  - {tid}")

In [ ]:
# Initialize agents
agents = {
    'zeroshot': ZeroShotAgent(verbose=False),
    # Add more agents here as they're implemented:
    # 'vqa_critic': VQACriticAgent(verbose=False),
}

print(f"Initialized {len(agents)} agent(s):")
for agent_slug in agents:
    print(f"  - {agent_slug}")

In [ ]:
# Run all edits in parallel
results = []

def apply_edit(template_id: str, spec_path: Path, agent_slug: str, agent) -> dict:
    """Apply a single edit and return result info."""
    output_dir = OUTPUT_ROOT / f"{template_id}-{agent_slug}"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_spec_path = output_dir / 'spec.json'
    output_render_path = output_dir / 'render.png'
    
    try:
        # Apply the edit
        result_path = agent.edit(
            spec_path=spec_path,
            instruction=EDIT_INSTRUCTION,
            output_path=output_spec_path
        )
        
        # Copy assets from original reconstruction directory if they exist
        import shutil
        reconstructions_dir = Path('../datasets/reconstructions') / template_id
        if reconstructions_dir.exists():
            # Copy all asset-*.png files
            for asset_file in reconstructions_dir.glob('asset-*.png'):
                shutil.copy2(asset_file, output_dir / asset_file.name)
            # Copy background.png if exists
            bg_file = reconstructions_dir / 'background.png'
            if bg_file.exists():
                shutil.copy2(bg_file, output_dir / 'background.png')
        
        # Render the edited spec
        with open(result_path, 'r') as f:
            import json
            spec_dict = json.load(f)
        
        spec = Spec(**spec_dict)
        render_image(
            spec=spec,
            output_path=output_render_path,
            canvas_width=spec.canvas_width,
            canvas_height=spec.canvas_height,
            asset_dir=output_dir  # Now has assets copied from reconstructions
        )
        
        return {
            'template_id': template_id,
            'agent': agent_slug,
            'status': 'success',
            'spec_path': str(result_path),
            'render_path': str(output_render_path),
            'error': None
        }
    except Exception as e:
        import traceback
        return {
            'template_id': template_id,
            'agent': agent_slug,
            'status': 'failed',
            'spec_path': None,
            'render_path': None,
            'error': str(e) + "\n" + traceback.format_exc()[:200]
        }

# Create all edit tasks
tasks = []
for _, row in test_specs.iterrows():
    for agent_slug, agent in agents.items():
        tasks.append({
            'template_id': row['template_id'],
            'spec_path': row['spec_path'],
            'agent_slug': agent_slug,
            'agent': agent
        })

print(f"Running {len(tasks)} edit tasks ({len(test_specs)} designs × {len(agents)} agents)...")
print()

# Execute in parallel
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {
        executor.submit(
            apply_edit,
            task['template_id'],
            task['spec_path'],
            task['agent_slug'],
            task['agent']
        ): task
        for task in tasks
    }
    
    for future in as_completed(futures):
        task = futures[future]
        result = future.result()
        results.append(result)
        
        status_icon = "✓" if result['status'] == 'success' else "✗"
        print(f"{status_icon} {result['template_id']} / {result['agent']}", end="")
        if result['status'] == 'failed':
            print(f" - Error: {result['error'][:80]}")
        else:
            print()

print()
print("Done!")

In [ ]:
# Summarize results
results_df = pd.DataFrame(results)

print("=" * 80)
print("SUMMARY")
print("=" * 80)
print()

# Overall stats
total = len(results_df)
success = len(results_df[results_df['status'] == 'success'])
failed = len(results_df[results_df['status'] == 'failed'])

print(f"Total edits: {total}")
print(f"Successful:  {success} ({100*success/total:.1f}%)")
print(f"Failed:      {failed} ({100*failed/total:.1f}%)")
print()

# Per-agent breakdown
print("Per-agent results:")
for agent_slug in agents.keys():
    agent_results = results_df[results_df['agent'] == agent_slug]
    agent_success = len(agent_results[agent_results['status'] == 'success'])
    agent_total = len(agent_results)
    print(f"  {agent_slug}: {agent_success}/{agent_total} successful")
print()

# Show failures if any
if failed > 0:
    print("Failed edits:")
    for _, row in results_df[results_df['status'] == 'failed'].iterrows():
        print(f"  {row['template_id']} / {row['agent']}:")
        print(f"    {row['error'][:200]}")
        print()

results_df

In [ ]:
# Show output directory structure
print(f"Edited specs saved to: {OUTPUT_ROOT.resolve()}")
print()
print("Directory structure:")
for output_dir in sorted(OUTPUT_ROOT.glob('*')):
    if output_dir.is_dir():
        spec_file = output_dir / 'spec.json'
        render_file = output_dir / 'render.png'
        status = "✓" if (spec_file.exists() and render_file.exists()) else "✗"
        print(f"  {status} {output_dir.name}/")
        if spec_file.exists():
            print(f"      spec.json ({spec_file.stat().st_size} bytes)")
        if render_file.exists():
            print(f"      render.png ({render_file.stat().st_size} bytes)")

In [ ]:
# Visualize rendered edits
import matplotlib.pyplot as plt
from PIL import Image

successful_results = results_df[results_df['status'] == 'success']

if len(successful_results) > 0:
    # Group by template_id to show all agent edits for each template
    for template_id in successful_results['template_id'].unique():
        template_results = successful_results[successful_results['template_id'] == template_id]
        
        # Load original for comparison
        original_spec_path = specs_df[specs_df['template_id'] == template_id]['spec_path'].iloc[0]
        original_render_path = Path('../datasets/canva') / f"{template_id}.webp"
        
        num_agents = len(template_results)
        fig, axes = plt.subplots(1, num_agents + 1, figsize=(4 * (num_agents + 1), 4))
        
        if num_agents == 0:
            axes = [axes]
        
        # Show original
        if original_render_path.exists():
            img = Image.open(original_render_path)
            axes[0].imshow(img)
            axes[0].set_title(f"Original\n{template_id}", fontsize=10)
            axes[0].axis('off')
        
        # Show each agent's edit
        for idx, (_, row) in enumerate(template_results.iterrows(), start=1):
            render_path = Path(row['render_path'])
            if render_path.exists():
                img = Image.open(render_path)
                axes[idx].imshow(img)
                axes[idx].set_title(f"{row['agent']}\n(edited)", fontsize=10)
                axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
else:
    print("No successful edits to visualize")